In [ ]:
import duckdb

con2 = duckdb.connect()
con2.execute("""
    CREATE OR REPLACE TABLE base_modelado_v13 AS
    SELECT * FROM read_parquet('C:/SERCOP_FINAL/03_resultados/base_modelado_v13.parquet')
""")
n_check = con2.execute("SELECT COUNT(*) FROM base_modelado_v13").fetchone()[0]
print(f"base_modelado_v13 recargada: {n_check:,} filas")

In [ ]:
import pandas as pd
import numpy as np

oferentes = pd.read_parquet(r"C:\SERCOP_FINAL\03_resultados\oferentes_consolidado.parquet")

# =========================================================
# GRUPO B: win_rate, loss_rate, perdedor recurrente
# =========================================================
resumen_proveedor = oferentes.groupby('oferente_ruc').agg(
    n_participaciones=('ocid', 'nunique'),
    n_ganadas=('es_ganador', 'sum')
).reset_index()

resumen_proveedor['win_rate'] = resumen_proveedor['n_ganadas'] / resumen_proveedor['n_participaciones']
resumen_proveedor['loss_rate'] = 1 - resumen_proveedor['win_rate']

print(f"Proveedores con historial de participación: {len(resumen_proveedor):,}")
print(resumen_proveedor['win_rate'].describe())

# Perdedor recurrente: participa mucho, casi nunca gana
UMBRAL_PARTICIPACIONES = 10
UMBRAL_WIN_RATE = 0.05

resumen_proveedor['perdedor_recurrente'] = (
    (resumen_proveedor['n_participaciones'] >= UMBRAL_PARTICIPACIONES) &
    (resumen_proveedor['win_rate'] <= UMBRAL_WIN_RATE)
)
print(f"\nPerdedores recurrentes (>={UMBRAL_PARTICIPACIONES} participaciones, win_rate<={UMBRAL_WIN_RATE}): {resumen_proveedor['perdedor_recurrente'].sum():,}")

resumen_proveedor.to_parquet(r"C:\SERCOP_FINAL\03_resultados\perfil_competitivo_proveedor.parquet", index=False)
print("Guardado: perfil_competitivo_proveedor.parquet")

In [ ]:
# --- Cruce contra las 170 fantasmas: ¿su win_rate es distinto? ---
rucs_fantasma = con2.execute("""
    SELECT DISTINCT proveedor_ruc FROM base_modelado_v13 WHERE proveedor_es_fantasma = TRUE
""").fetchdf()['proveedor_ruc'].tolist()

resumen_proveedor['es_fantasma'] = resumen_proveedor['oferente_ruc'].isin(rucs_fantasma)

cruce_winrate = resumen_proveedor.groupby('es_fantasma').agg(
    n=('win_rate', 'size'),
    media_winrate=('win_rate', 'mean'),
    mediana_winrate=('win_rate', 'median'),
    media_participaciones=('n_participaciones', 'mean'),
    n_perdedor_recurrente=('perdedor_recurrente', 'sum'),
)
print(cruce_winrate)

In [ ]:
from itertools import combinations
from collections import defaultdict

UMBRAL_MAX_OFERENTES = 30

# Necesitamos, por proceso: lista de (ruc, es_ganador)
oferentes_validos = oferentes[oferentes['oferente_ruc'].notna()].copy()
grupos_completos = oferentes_validos.groupby('ocid').apply(
    lambda g: list(zip(g['oferente_ruc'], g['es_ganador']))
)
grupos_completos = grupos_completos[grupos_completos.apply(len).between(2, UMBRAL_MAX_OFERENTES)]

print(f"Procesos usados: {len(grupos_completos):,}")

# Para cada par (a,b) que compiten juntos: cuántas veces gana cada uno
dominancia = defaultdict(lambda: {'coincidencias': 0, 'gana_a': 0, 'gana_b': 0})

for participantes in grupos_completos:
    rucs = sorted(set(r for r, _ in participantes))
    ganador_ruc = next((r for r, g in participantes if g), None)
    for a, b in combinations(rucs, 2):
        key = (a, b)
        dominancia[key]['coincidencias'] += 1
        if ganador_ruc == a:
            dominancia[key]['gana_a'] += 1
        elif ganador_ruc == b:
            dominancia[key]['gana_b'] += 1

df_dominancia = pd.DataFrame([
    {'ruc_a': a, 'ruc_b': b, **v} for (a, b), v in dominancia.items()
])
df_dominancia = df_dominancia[df_dominancia['coincidencias'] >= 3]  # descartar pares con muy pocas observaciones
df_dominancia['dominancia_a'] = df_dominancia['gana_a'] / df_dominancia['coincidencias']
df_dominancia['dominancia_b'] = df_dominancia['gana_b'] / df_dominancia['coincidencias']
df_dominancia['dominancia_max'] = df_dominancia[['dominancia_a', 'dominancia_b']].max(axis=1)

print(f"\nPares con >=3 coincidencias: {len(df_dominancia):,}")
print(df_dominancia['dominancia_max'].describe())

# --- Pares donde participa una fantasma, con dominancia alta ---
pares_fantasma_dom = df_dominancia[
    df_dominancia['ruc_a'].isin(rucs_fantasma) | df_dominancia['ruc_b'].isin(rucs_fantasma)
]
print(f"\nPares con fantasma (>=3 coincidencias): {len(pares_fantasma_dom):,}")
print(pares_fantasma_dom.sort_values('dominancia_max', ascending=False).head(15))

df_dominancia.to_parquet(r"C:\SERCOP_FINAL\03_resultados\dominancia_pares.parquet", index=False)
print("\nGuardado: dominancia_pares.parquet")

In [ ]:
# --- Comparación formal: dominancia_max en pares con fantasma vs. resto ---
resto = df_dominancia[
    ~(df_dominancia['ruc_a'].isin(rucs_fantasma) | df_dominancia['ruc_b'].isin(rucs_fantasma))
]

print("Dominancia máxima - pares CON fantasma:")
print(pares_fantasma_dom['dominancia_max'].describe())

print("\nDominancia máxima - pares SIN fantasma (resto):")
print(resto['dominancia_max'].describe())

# --- Enriquecimiento en la cola extrema ---
for umbral in [0.6, 0.7, 0.8, 0.9, 1.0]:
    pct_fantasma = (pares_fantasma_dom['dominancia_max'] >= umbral).mean() * 100
    pct_resto = (resto['dominancia_max'] >= umbral).mean() * 100
    print(f"dominancia_max >= {umbral}: fantasma={pct_fantasma:.1f}%  resto={pct_resto:.1f}%  lift={pct_fantasma/pct_resto if pct_resto>0 else float('nan'):.2f}x")

# --- Caso destacado: RUC_OCULTO_17 (ya lo vimos antes con PROVEEDOR_OCULTO) pierde consistentemente contra varios rivales ---
caso_923858856 = df_dominancia[
    (df_dominancia['ruc_a'] == 'RUC_OCULTO_02') | (df_dominancia['ruc_b'] == 'RUC_OCULTO_02')
].sort_values('coincidencias', ascending=False)
print("\nTodos los pares de RUC_OCULTO_02 (>=3 coincidencias):")
print(caso_923858856)

In [ ]:
# =========================================================
# Agregación a nivel de proveedor: ¿qué tan "extremas" son sus relaciones par a par?
# =========================================================

def agregar_dominancia(ruc, df):
    filas = df[(df['ruc_a'] == ruc) | (df['ruc_b'] == ruc)]
    if len(filas) == 0:
        return pd.Series({'dominancia_maxima_historica': np.nan, 'dominancia_promedio_historica': np.nan,
                           'n_pares_evaluados': 0, 'n_pares_dominancia_alta': 0})
    return pd.Series({
        'dominancia_maxima_historica': filas['dominancia_max'].max(),
        'dominancia_promedio_historica': filas['dominancia_max'].mean(),
        'n_pares_evaluados': len(filas),
        'n_pares_dominancia_alta': (filas['dominancia_max'] >= 0.8).sum(),
    })

rucs_unicos = pd.concat([df_dominancia['ruc_a'], df_dominancia['ruc_b']]).unique()
print(f"Proveedores con al menos 1 par evaluado: {len(rucs_unicos):,}")

# Nota: esto puede tardar unos minutos por el filtrado repetido; si es muy lento, avísame y lo optimizamos con groupby
perfil_dominancia = pd.DataFrame({'proveedor_ruc': rucs_unicos})
resultado = perfil_dominancia['proveedor_ruc'].apply(lambda r: agregar_dominancia(r, df_dominancia))
perfil_dominancia = pd.concat([perfil_dominancia, resultado], axis=1)

perfil_dominancia['es_fantasma'] = perfil_dominancia['proveedor_ruc'].isin(rucs_fantasma)

print("\nComparación a nivel de proveedor:")
print(perfil_dominancia.groupby('es_fantasma')[['dominancia_maxima_historica', 'dominancia_promedio_historica', 'n_pares_dominancia_alta']].describe())

perfil_dominancia.to_parquet(r"C:\SERCOP_FINAL\03_resultados\perfil_dominancia_proveedor.parquet", index=False)
print("\nGuardado: perfil_dominancia_proveedor.parquet")

In [ ]:
con2.execute("""
    CREATE OR REPLACE TABLE base_modelado_v14 AS
    SELECT * FROM read_parquet('C:/SERCOP_FINAL/03_resultados/base_modelado_v14.parquet')
""")
n_check = con2.execute("SELECT COUNT(*) FROM base_modelado_v14").fetchone()[0]
print(f"base_modelado_v14 recargada: {n_check:,} filas")

In [ ]:
# =========================================================
# Reconstruir Modelo A de Isolation Forest con dominancia + precio sin fuga
# =========================================================
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

df_a2 = con2.execute("""
    SELECT
        ocid, proveedor_ruc, anio_proceso,
        tiene_periodo_competitivo,
        desviacion_precio_log_z_sinfuga,
        concentracion_entidad_proveedor,
        proveedor_antiguedad_dias,
        proveedor_es_entidad_publica,
        proveedor_contrato_tras_suspension,
        proveedor_provincia_distinta_entidad,
        proveedor_es_fantasma
    FROM base_modelado_v14
    WHERE concentracion_entidad_proveedor IS NOT NULL
""").fetchdf()

df_a2 = df_a2.merge(
    perfil_dominancia[['proveedor_ruc', 'dominancia_maxima_historica', 'n_pares_dominancia_alta']],
    on='proveedor_ruc', how='left'
)

print(f"Filas: {len(df_a2):,}")
print(f"Sin datos de dominancia (proveedor sin pares evaluados): {df_a2['dominancia_maxima_historica'].isna().sum():,}")

df_a2['dominancia_desconocida'] = df_a2['dominancia_maxima_historica'].isna().astype(int)
df_a2['dominancia_maxima_historica'] = df_a2['dominancia_maxima_historica'].fillna(-1)
df_a2['n_pares_dominancia_alta'] = df_a2['n_pares_dominancia_alta'].fillna(0)

df_a2['antiguedad_desconocida'] = df_a2['proveedor_antiguedad_dias'].isna().astype(int)
df_a2['proveedor_antiguedad_dias_num'] = df_a2['proveedor_antiguedad_dias'].fillna(-1)
df_a2['desviacion_precio_log_z_sinfuga'] = df_a2['desviacion_precio_log_z_sinfuga'].fillna(0)

for col in ['tiene_periodo_competitivo', 'proveedor_es_entidad_publica',
            'proveedor_contrato_tras_suspension', 'proveedor_provincia_distinta_entidad']:
    df_a2[col] = df_a2[col].astype(int)

features_a2 = [
    'desviacion_precio_log_z_sinfuga', 'concentracion_entidad_proveedor',
    'proveedor_antiguedad_dias_num', 'antiguedad_desconocida',
    'proveedor_es_entidad_publica', 'proveedor_contrato_tras_suspension',
    'proveedor_provincia_distinta_entidad', 'tiene_periodo_competitivo',
    'dominancia_maxima_historica', 'n_pares_dominancia_alta', 'dominancia_desconocida',
]

X_a2 = df_a2[features_a2].values
clf_a2 = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
clf_a2.fit(X_a2)

df_a2['anomaly_score_a2'] = -clf_a2.score_samples(X_a2)
df_a2['percentil_score_a2'] = df_a2['anomaly_score_a2'].rank(pct=True) * 100

auc_a2 = roc_auc_score(df_a2['proveedor_es_fantasma'].astype(int), df_a2['anomaly_score_a2'])
print(f"\nAUC-ROC Modelo A2 (con dominancia + precio sin fuga): {auc_a2:.4f}")
print("(Modelo A original: 0.7393)")

for k in [10, 50, 100, 200, 500]:
    top_k = df_a2.nlargest(k, 'anomaly_score_a2')
    capturadas = top_k['proveedor_es_fantasma'].sum()
    print(f"Top {k}: {capturadas} de 170 fantasmas")

df_a2.to_parquet(r"C:\SERCOP_FINAL\03_resultados\scores_modelo_a2.parquet", index=False)
print("\nGuardado: scores_modelo_a2.parquet")


In [ ]:
import xgboost as xgb
from sklearn.metrics import average_precision_score

In [ ]:
df_m3b = con2.execute("""
    SELECT
        ocid, proveedor_ruc, anio_proceso,
        tiene_periodo_competitivo,
        desviacion_precio_log_z_sinfuga,
        concentracion_entidad_proveedor,
        proveedor_antiguedad_dias,
        proveedor_es_entidad_publica,
        proveedor_contrato_tras_suspension,
        proveedor_provincia_distinta_entidad,
        proveedor_unico,
        n_oferentes,
        proveedor_es_fantasma
    FROM base_modelado_v14
    WHERE concentracion_entidad_proveedor IS NOT NULL
""").fetchdf()

df_m3b = df_m3b.merge(
    perfil_dominancia[['proveedor_ruc', 'dominancia_maxima_historica', 'n_pares_dominancia_alta']],
    on='proveedor_ruc', how='left'
)

df_m3b['dominancia_desconocida'] = df_m3b['dominancia_maxima_historica'].isna().astype(int)
df_m3b['dominancia_maxima_historica'] = df_m3b['dominancia_maxima_historica'].fillna(-1)
df_m3b['n_pares_dominancia_alta'] = df_m3b['n_pares_dominancia_alta'].fillna(0)
df_m3b['antiguedad_desconocida'] = df_m3b['proveedor_antiguedad_dias'].isna().astype(int)
df_m3b['proveedor_antiguedad_dias_num'] = df_m3b['proveedor_antiguedad_dias'].fillna(-1)
df_m3b['desviacion_precio_log_z_sinfuga'] = df_m3b['desviacion_precio_log_z_sinfuga'].fillna(0)
df_m3b['proveedor_unico_desconocido'] = df_m3b['proveedor_unico'].isna().astype(int)
df_m3b['proveedor_unico_num'] = df_m3b['proveedor_unico'].fillna(False).astype(int)
df_m3b['n_oferentes'] = df_m3b['n_oferentes'].fillna(0)

for col in ['tiene_periodo_competitivo', 'proveedor_es_entidad_publica',
            'proveedor_contrato_tras_suspension', 'proveedor_provincia_distinta_entidad']:
    df_m3b[col] = df_m3b[col].astype(int)

features_m3b = [
    'desviacion_precio_log_z_sinfuga', 'concentracion_entidad_proveedor',
    'proveedor_antiguedad_dias_num', 'antiguedad_desconocida',
    'proveedor_es_entidad_publica', 'proveedor_contrato_tras_suspension',
    'proveedor_provincia_distinta_entidad', 'tiene_periodo_competitivo',
    'proveedor_unico_num', 'proveedor_unico_desconocido', 'n_oferentes',
    'dominancia_maxima_historica', 'n_pares_dominancia_alta', 'dominancia_desconocida',
]

y_m3b = df_m3b['proveedor_es_fantasma'].astype(int)
ANIO_CORTE = 2023
train_mask = df_m3b['anio_proceso'] < ANIO_CORTE
test_mask = df_m3b['anio_proceso'] >= ANIO_CORTE

X_train, y_train = df_m3b.loc[train_mask, features_m3b], y_m3b[train_mask]
X_test, y_test = df_m3b.loc[test_mask, features_m3b], y_m3b[test_mask]
print(f"Train: {len(X_train):,} filas, {y_train.sum()} fantasmas | Test: {len(X_test):,} filas, {y_test.sum()} fantasmas")

scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
modelo_xgb2 = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, eval_metric='aucpr', random_state=42, n_jobs=-1,
)
modelo_xgb2.fit(X_train, y_train)

y_score_test = modelo_xgb2.predict_proba(X_test)[:, 1]
auc_roc2 = roc_auc_score(y_test, y_score_test)
auc_pr2 = average_precision_score(y_test, y_score_test)
print(f"\nAUC-ROC (test temporal, con dominancia): {auc_roc2:.4f}   (versión anterior: 0.4333)")
print(f"PR-AUC: {auc_pr2:.4f}")

for k in [10, 50, 100, 200, 500, 1000]:
    top_k = pd.Series(y_score_test, index=y_test.index).sort_values(ascending=False).head(k)
    capturados = y_test.loc[top_k.index].sum()
    print(f"Top {k}: {capturados} de {int(y_test.sum())} fantasmas reales")

importancias2 = pd.Series(modelo_xgb2.feature_importances_, index=features_m3b).sort_values(ascending=False)
print("\nImportancia de variables:")
print(importancias2)

In [ ]:
distribucion_fantasma_anio = con2.execute("""
    SELECT EXTRACT(YEAR FROM fecha_release) AS anio, COUNT(*) AS n_procesos_fantasma
    FROM base_modelado_v14
    WHERE proveedor_es_fantasma = TRUE
    GROUP BY 1 ORDER BY 1
""").fetchdf()
print(distribucion_fantasma_anio)

In [ ]:
RUTA_FANTASMAS = r"C:\SERCOP_FINAL\05_documentos\SRI_Catastro_Empresas_Fantasmas.csv"

df_fantasmas = pd.read_csv(RUTA_FANTASMAS, delimiter='|', encoding='latin-1')
df_fantasmas['numero_ruc_str'] = df_fantasmas['NUMERO_RUC'].apply(
    lambda x: str(int(x)).zfill(13) if pd.notna(x) else None
)

con2.execute("""
    CREATE OR REPLACE TABLE empresas_fantasmas AS
    SELECT DISTINCT numero_ruc_str AS ruc, ESTADO_RUC, DETALLE_CALIFICACION,
           FECHA_INICIO_CALIFICACION, FECHA_NOTIFICACION_OFICIO
    FROM df_fantasmas
    WHERE numero_ruc_str IS NOT NULL
""")

n_check = con2.execute("SELECT COUNT(*) FROM empresas_fantasmas").fetchone()[0]
print(f"empresas_fantasmas recargada: {n_check:,} RUC")

In [ ]:
# --- ¿Cuánto tarda el SRI en calificar a una empresa como fantasma, desde que empezó a operar? ---
rezago_calificacion = con2.execute("""
    SELECT
        ef.ruc,
        MIN(b.fecha_release) AS primer_contrato_sercop,
        MAX(b.fecha_release) AS ultimo_contrato_sercop
    FROM base_modelado_v14 b
    JOIN empresas_fantasmas ef ON b.proveedor_ruc = ef.ruc
    GROUP BY 1
""").fetchdf()
print(rezago_calificacion)

# Fecha de calificación real desde el archivo original (si la guardaste con esas columnas)
print(df_fantasmas[df_fantasmas['numero_ruc_str'].isin(rezago_calificacion['ruc'])][
    ['numero_ruc_str', 'FECHA_INICIO_CALIFICACION', 'FECHA_NOTIFICACION_OFICIO']
])

In [ ]:
# --- Convertir fechas seriales de Excel a fechas reales ---
df_fantasmas['fecha_inicio_calificacion_real'] = pd.to_datetime('1899-12-30') + pd.to_timedelta(
    pd.to_numeric(df_fantasmas['FECHA_INICIO_CALIFICACION'], errors='coerce'), unit='D'
)
df_fantasmas['fecha_notificacion_oficio_real'] = pd.to_datetime('1899-12-30') + pd.to_timedelta(
    pd.to_numeric(df_fantasmas['FECHA_NOTIFICACION_OFICIO'], errors='coerce'), unit='D'
)

fechas_fantasmas = df_fantasmas[df_fantasmas['numero_ruc_str'].isin(rezago_calificacion['ruc'])][
    ['numero_ruc_str', 'fecha_inicio_calificacion_real', 'fecha_notificacion_oficio_real']
].drop_duplicates()

rezago_completo = rezago_calificacion.merge(
    fechas_fantasmas, left_on='ruc', right_on='numero_ruc_str'
)

rezago_completo['primer_contrato_sercop'] = pd.to_datetime(rezago_completo['primer_contrato_sercop'])
rezago_completo['rezago_dias'] = (
    rezago_completo['fecha_notificacion_oficio_real'] - rezago_completo['primer_contrato_sercop']
).dt.days
rezago_completo['rezago_anios'] = rezago_completo['rezago_dias'] / 365.25

print(rezago_completo[['ruc', 'primer_contrato_sercop', 'fecha_notificacion_oficio_real', 'rezago_anios']])
print(f"\nRezago promedio: {rezago_completo['rezago_anios'].mean():.1f} años")
print(f"Rezago mediano: {rezago_completo['rezago_anios'].median():.1f} años")

In [ ]:
import os
os.chdir(r'C:\SERCOP_FINAL')  # ajusta si tu carpeta raíz es otra
print(os.getcwd())
print(os.path.exists('03_resultados/base_modelado_v14.parquet'))

In [ ]:
import glob
print(glob.glob(r'C:\SERCOP_FINAL\03_resultados\base_modelado_v14*'))

In [ ]:
import duckdb, pandas as pd, numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")
con2.execute("CREATE OR REPLACE TABLE perfil_dominancia_proveedor AS SELECT * FROM read_parquet('03_resultados/perfil_dominancia_proveedor.parquet')")
con2.execute("CREATE OR REPLACE TABLE perfil_competitivo_proveedor AS SELECT * FROM read_parquet('03_resultados/perfil_competitivo_proveedor.parquet')")

# Join base + dominancia + win_rate (igual que en Modelo 3 v2)
base_m3 = con2.execute("""
    SELECT b.*,
           d.dominancia_maxima_historica, d.dominancia_promedio_historica,
           d.n_pares_evaluados, d.n_pares_dominancia_alta,
           c.win_rate, c.loss_rate, c.n_participaciones, c.perdedor_recurrente
    FROM base_modelado_v14 b
    LEFT JOIN perfil_dominancia_proveedor d ON b.proveedor_ruc = d.proveedor_ruc
    LEFT JOIN perfil_competitivo_proveedor c ON b.proveedor_ruc = c.oferente_ruc
""").df()

# Flags de nulos (nunca imputar con valor típico)
for col in ['dominancia_maxima_historica','dominancia_promedio_historica','n_pares_dominancia_alta',
            'win_rate','loss_rate','n_participaciones']:
    base_m3[f'{col}_desconocido'] = base_m3[col].isna().astype(int)
    base_m3[col] = base_m3[col].fillna(-1)
base_m3['perdedor_recurrente'] = base_m3['perdedor_recurrente'].fillna(False).astype(int)

# CORTE TEMPORAL CORREGIDO: excluye 2020+ de la evaluación formal

train_mask = base_m3['anio_proceso'].between(2015, 2017)
test_mask  = base_m3['anio_proceso'].between(2018, 2019)
no_evaluado_mask = base_m3['anio_proceso'] >= 2020

print("Train (2015-2017):", train_mask.sum(), "filas |", base_m3.loc[train_mask,'proveedor_es_fantasma'].sum(), "fantasmas")
print("Test  (2018-2019):", test_mask.sum(), "filas |", base_m3.loc[test_mask,'proveedor_es_fantasma'].sum(), "fantasmas")
print("No evaluado (2020+, ground truth no confiable):", no_evaluado_mask.sum(), "filas |", base_m3.loc[no_evaluado_mask,'proveedor_es_fantasma'].sum(), "fantasmas confirmados (puede haber más sin detectar)")

features = ['desviacion_precio_log_z_sinfuga','n_oferentes','proveedor_unico',
            'proveedor_unico_desconocido','dominancia_maxima_historica','dominancia_promedio_historica',
            'n_pares_dominancia_alta','win_rate','loss_rate','n_participaciones','perdedor_recurrente',
            'dominancia_maxima_historica_desconocido','win_rate_desconocido']
features = [f for f in features if f in base_m3.columns]

X_train, y_train = base_m3.loc[train_mask, features], base_m3.loc[train_mask, 'proveedor_es_fantasma']
X_test,  y_test  = base_m3.loc[test_mask, features],  base_m3.loc[test_mask, 'proveedor_es_fantasma']

scale = (y_train==0).sum() / max((y_train==1).sum(), 1)
modelo3_v3 = xgb.XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=scale,
                                eval_metric='aucpr', random_state=42)
modelo3_v3.fit(X_train, y_train)

scores_test = modelo3_v3.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, scores_test)
print(f"\nAUC-ROC Modelo 3 v3 (corte 2015-2017/2018-2019, sin 2020+): {auc:.4f}")

top_n = base_m3.loc[test_mask].assign(score=scores_test).sort_values('score', ascending=False)
for k in [10,50,100,500]:
    capturados = top_n.head(k)['proveedor_es_fantasma'].sum()
    print(f"Top {k}: {capturados} de {y_test.sum()} fantasmas")

importancias = pd.Series(modelo3_v3.feature_importances_, index=features).sort_values(ascending=False)
print("\nImportancia de variables:\n", importancias)



In [ ]:
print(con2.execute("DESCRIBE perfil_dominancia_proveedor").df())
print(con2.execute("DESCRIBE perfil_competitivo_proveedor").df())

In [ ]:
print([c for c in base_m3.columns if 'fantasma' in c.lower()])

In [ ]:
import duckdb, pandas as pd

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE oferentes_consolidado AS SELECT * FROM read_parquet('03_resultados/oferentes_consolidado.parquet')")

# oferentes_consolidado tiene: ocid, oferente_ruc, oferente_nombre, es_ganador, anio
# Necesitamos también el anio de CADA proceso para poder filtrar "anteriores a Y"

anios = range(2016, 2027)  # 2015 no tiene años previos, se excluye igual que con el precio
resultados_winrate = []

for anio_corte in anios:
    stats_anio = con2.execute(f"""
        SELECT
            oferente_ruc,
            COUNT(*) AS n_participaciones,
            SUM(CASE WHEN es_ganador THEN 1 ELSE 0 END) AS n_ganadas,
            CAST(SUM(CASE WHEN es_ganador THEN 1 ELSE 0 END) AS DOUBLE) / COUNT(*) AS win_rate,
            CAST(SUM(CASE WHEN NOT es_ganador THEN 1 ELSE 0 END) AS DOUBLE) / COUNT(*) AS loss_rate,
            {anio_corte} AS anio_aplicable
        FROM oferentes_consolidado
        WHERE anio < {anio_corte} AND oferente_ruc IS NOT NULL
        GROUP BY oferente_ruc
    """).df()
    resultados_winrate.append(stats_anio)
    print(f"Año {anio_corte}: {len(stats_anio)} proveedores con historial previo")

winrate_expandido = pd.concat(resultados_winrate, ignore_index=True)
winrate_expandido['perdedor_recurrente'] = (
    (winrate_expandido['n_participaciones'] >= 10) & (winrate_expandido['win_rate'] <= 0.05)
)
winrate_expandido.to_parquet('03_resultados/winrate_expandido.parquet', index=False)
print("\nwinrate_expandido guardado:", len(winrate_expandido), "filas")
print(winrate_expandido.head())

In [ ]:
import duckdb, pandas as pd, time

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE oferentes_consolidado AS SELECT * FROM read_parquet('03_resultados/oferentes_consolidado.parquet')")

anios = range(2016, 2027)
resultados_dominancia = []

for anio_corte in anios:
    t0 = time.time()
    con2.execute(f"""
        CREATE OR REPLACE TEMP TABLE procesos_validos_corte AS
        SELECT ocid, COUNT(*) AS n_ofertantes
        FROM oferentes_consolidado
        WHERE anio < {anio_corte} AND oferente_ruc IS NOT NULL
        GROUP BY ocid
        HAVING COUNT(*) <= 30
    """)

    pares = con2.execute(f"""
        WITH participantes AS (
            SELECT o.ocid, o.oferente_ruc, o.es_ganador
            FROM oferentes_consolidado o
            JOIN procesos_validos_corte p ON o.ocid = p.ocid
            WHERE o.anio < {anio_corte} AND o.oferente_ruc IS NOT NULL
        )
        SELECT
            a.oferente_ruc AS ruc_a,
            b.oferente_ruc AS ruc_b,
            COUNT(*) AS coincidencias,
            SUM(CASE WHEN a.es_ganador THEN 1 ELSE 0 END) AS gana_a,
            SUM(CASE WHEN b.es_ganador THEN 1 ELSE 0 END) AS gana_b,
            {anio_corte} AS anio_aplicable
        FROM participantes a
        JOIN participantes b ON a.ocid = b.ocid AND a.oferente_ruc < b.oferente_ruc
        GROUP BY a.oferente_ruc, b.oferente_ruc
        HAVING COUNT(*) >= 3
    """).df()

    resultados_dominancia.append(pares)
    print(f"Año {anio_corte}: {len(pares)} pares con >=3 coincidencias ({time.time()-t0:.1f}s)")

dominancia_expandida = pd.concat(resultados_dominancia, ignore_index=True)
dominancia_expandida['dominancia_a'] = dominancia_expandida['gana_a'] / dominancia_expandida['coincidencias']
dominancia_expandida['dominancia_b'] = dominancia_expandida['gana_b'] / dominancia_expandida['coincidencias']
dominancia_expandida['dominancia_max'] = dominancia_expandida[['dominancia_a','dominancia_b']].max(axis=1)
dominancia_expandida.to_parquet('03_resultados/dominancia_pares_expandido.parquet', index=False)
print("\nGuardado. Total filas:", len(dominancia_expandida))

In [ ]:
import duckdb, pandas as pd, numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE dominancia_pares_expandido AS SELECT * FROM read_parquet('03_resultados/dominancia_pares_expandido.parquet')")
con2.execute("CREATE OR REPLACE TABLE winrate_expandido AS SELECT * FROM read_parquet('03_resultados/winrate_expandido.parquet')")
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")

# Long format: cada proveedor aparece como ruc_a o ruc_b -> un registro por rol
perfil_dominancia_expandido = con2.execute("""
    WITH largo AS (
        SELECT ruc_a AS proveedor_ruc, dominancia_a AS dominancia, anio_aplicable FROM dominancia_pares_expandido
        UNION ALL
        SELECT ruc_b AS proveedor_ruc, dominancia_b AS dominancia, anio_aplicable FROM dominancia_pares_expandido
    )
    SELECT
        proveedor_ruc,
        anio_aplicable,
        MAX(dominancia) AS dominancia_maxima_historica,
        AVG(dominancia) AS dominancia_promedio_historica,
        COUNT(*) AS n_pares_evaluados,
        SUM(CASE WHEN dominancia >= 0.6 THEN 1 ELSE 0 END) AS n_pares_dominancia_alta
    FROM largo
    GROUP BY proveedor_ruc, anio_aplicable
""").df()
perfil_dominancia_expandido.to_parquet('03_resultados/perfil_dominancia_expandido.parquet', index=False)
con2.execute("CREATE OR REPLACE TABLE perfil_dominancia_expandido AS SELECT * FROM perfil_dominancia_expandido")
print("perfil_dominancia_expandido:", len(perfil_dominancia_expandido), "filas")

# Join final: proveedor_ruc + anio_proceso = anio_aplicable (info estrictamente previa al proceso)
base_m3 = con2.execute("""
    SELECT b.*,
           d.dominancia_maxima_historica, d.dominancia_promedio_historica,
           d.n_pares_evaluados, d.n_pares_dominancia_alta,
           w.win_rate, w.loss_rate, w.n_participaciones, w.perdedor_recurrente
    FROM base_modelado_v14 b
    LEFT JOIN perfil_dominancia_expandido d
        ON b.proveedor_ruc = d.proveedor_ruc AND b.anio_proceso = d.anio_aplicable
    LEFT JOIN winrate_expandido w
        ON b.proveedor_ruc = w.oferente_ruc AND b.anio_proceso = w.anio_aplicable
""").df()

for col in ['dominancia_maxima_historica','dominancia_promedio_historica','n_pares_dominancia_alta',
            'win_rate','loss_rate','n_participaciones']:
    base_m3[f'{col}_desconocido'] = base_m3[col].isna().astype(int)
    base_m3[col] = base_m3[col].fillna(-1)
base_m3['perdedor_recurrente'] = base_m3['perdedor_recurrente'].fillna(False).astype(int)

train_mask = base_m3['anio_proceso'].between(2015, 2017)
test_mask  = base_m3['anio_proceso'].between(2018, 2019)

print("Train (2015-2017):", train_mask.sum(), "filas |", base_m3.loc[train_mask,'proveedor_es_fantasma'].sum(), "fantasmas")
print("Test  (2018-2019):", test_mask.sum(), "filas |", base_m3.loc[test_mask,'proveedor_es_fantasma'].sum(), "fantasmas")

features = ['desviacion_precio_log_z_sinfuga','n_oferentes','proveedor_unico',
            'proveedor_unico_desconocido','dominancia_maxima_historica','dominancia_promedio_historica',
            'n_pares_dominancia_alta','win_rate','loss_rate','n_participaciones','perdedor_recurrente',
            'dominancia_maxima_historica_desconocido','win_rate_desconocido']
features = [f for f in features if f in base_m3.columns]

X_train, y_train = base_m3.loc[train_mask, features], base_m3.loc[train_mask, 'proveedor_es_fantasma']
X_test,  y_test  = base_m3.loc[test_mask, features],  base_m3.loc[test_mask, 'proveedor_es_fantasma']

scale = (y_train==0).sum() / max((y_train==1).sum(), 1)
modelo3_v4 = xgb.XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=scale,
                                eval_metric='aucpr', random_state=42)
modelo3_v4.fit(X_train, y_train)

scores_test = modelo3_v4.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, scores_test)
print(f"\nAUC-ROC Modelo 3 v4 (dominancia/win_rate SIN fuga temporal): {auc:.4f}")

top_n = base_m3.loc[test_mask].assign(score=scores_test).sort_values('score', ascending=False)
for k in [10,50,100,500]:
    capturados = top_n.head(k)['proveedor_es_fantasma'].sum()
    print(f"Top {k}: {capturados} de {y_test.sum()} fantasmas")

importancias = pd.Series(modelo3_v4.feature_importances_, index=features).sort_values(ascending=False)
print("\nImportancia de variables:\n", importancias)

In [ ]:
# Dónde caen realmente los 29 fantasmas en el ranking (percentil, no solo top-K fijo)
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, scores_test)
print(f"PR-AUC (average precision): {pr_auc:.4f}  (línea base = {y_test.mean():.5f} por prevalencia)")

top_n_full = base_m3.loc[test_mask].assign(score=scores_test).sort_values('score', ascending=False).reset_index(drop=True)
top_n_full['posicion'] = top_n_full.index + 1
top_n_full['percentil'] = top_n_full['posicion'] / len(top_n_full)

fantasmas_test = top_n_full[top_n_full['proveedor_es_fantasma'] == True]
print(f"\nPosición/percentil de los {len(fantasmas_test)} fantasmas en el ranking del test:")
print(fantasmas_test[['proveedor_ruc','score','posicion','percentil']].sort_values('posicion').to_string(index=False))

print(f"\nPercentil promedio de los fantasmas: {fantasmas_test['percentil'].mean():.4f}")
print(f"Percentil mediano de los fantasmas: {fantasmas_test['percentil'].median():.4f}")

In [ ]:
import matplotlib.pyplot as plt

# 1) Agregación a nivel proveedor: mejor (máximo) score por RUC en el test
resumen_proveedor = top_n_full.groupby('proveedor_ruc').agg(
    score_maximo=('score','max'),
    percentil_minimo=('percentil','min'),
    n_procesos_test=('score','size'),
    es_fantasma=('proveedor_es_fantasma','max')
).reset_index().sort_values('score_maximo', ascending=False)

n_proveedores_unicos = resumen_proveedor.shape[0]
n_fantasmas_unicos = resumen_proveedor['es_fantasma'].sum()
print(f"Proveedores únicos en test: {n_proveedores_unicos} | fantasmas únicos: {n_fantasmas_unicos}")

for k in [10, 50, 100, 500, 1000]:
    capturados = resumen_proveedor.head(k)['es_fantasma'].sum()
    print(f"Top {k} proveedores (por mejor score): {capturados} de {n_fantasmas_unicos} fantasmas únicos")

print("\nPercentil mínimo (mejor ranking alcanzado) de cada fantasma único:")
print(resumen_proveedor[resumen_proveedor['es_fantasma']][['proveedor_ruc','score_maximo','percentil_minimo','n_procesos_test']].to_string(index=False))

# 2) Gráfico: distribución de percentiles, fantasma vs resto
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(top_n_full.loc[~top_n_full['proveedor_es_fantasma'], 'percentil'], bins=50, alpha=0.5, label='No fantasma', density=True, color='steelblue')
ax.hist(top_n_full.loc[top_n_full['proveedor_es_fantasma'], 'percentil'], bins=50, alpha=0.7, label='Fantasma (SRI)', density=True, color='crimson')
ax.set_xlabel('Percentil en el ranking del modelo (0 = score más alto)')
ax.set_ylabel('Densidad')
ax.set_title('Modelo 3 v4 (sin fuga): posición de fantasmas SRI vs resto en el test 2018-2019')
ax.legend()
plt.tight_layout()
plt.savefig('03_resultados/grafico_modelo3_percentiles.png', dpi=150)
plt.show()
print("Gráfico guardado en 03_resultados/grafico_modelo3_percentiles.png")

kkkkkkkkkkkkk   
# propuesta de Santiago (competidor habitual, entropía de ganadores, entrada/salida de proveedores por mercado-año  

In [ ]:
import duckdb, pandas as pd

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE dominancia_pares_expandido AS SELECT * FROM read_parquet('03_resultados/dominancia_pares_expandido.parquet')")

competidor_habitual = con2.execute("""
    WITH largo AS (
        SELECT ruc_a AS proveedor_ruc, ruc_b AS rival_ruc, coincidencias, anio_aplicable FROM dominancia_pares_expandido
        UNION ALL
        SELECT ruc_b AS proveedor_ruc, ruc_a AS rival_ruc, coincidencias, anio_aplicable FROM dominancia_pares_expandido
    ),
    agregado AS (
        SELECT proveedor_ruc, anio_aplicable,
               SUM(coincidencias) AS total_coincidencias,
               MAX(coincidencias) AS max_coincidencias_un_rival,
               COUNT(DISTINCT rival_ruc) AS n_rivales_distintos
        FROM largo
        GROUP BY proveedor_ruc, anio_aplicable
    )
    SELECT *,
           CAST(max_coincidencias_un_rival AS DOUBLE) / total_coincidencias AS concentracion_rival_habitual
    FROM agregado
""").df()

competidor_habitual.to_parquet('03_resultados/competidor_habitual.parquet', index=False)
print("Filas:", len(competidor_habitual))
print(competidor_habitual['concentracion_rival_habitual'].describe())

# Comparación contra fantasmas conocidos
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")
con2.execute("CREATE OR REPLACE TABLE competidor_habitual AS SELECT * FROM competidor_habitual")

comparacion = con2.execute("""
    SELECT b.proveedor_es_fantasma,
           AVG(c.concentracion_rival_habitual) AS concentracion_promedio,
           MEDIAN(c.concentracion_rival_habitual) AS concentracion_mediana,
           COUNT(*) AS n
    FROM base_modelado_v14 b
    JOIN competidor_habitual c ON b.proveedor_ruc = c.proveedor_ruc AND b.anio_proceso = c.anio_aplicable
    GROUP BY b.proveedor_es_fantasma
""").df()
print(comparacion)

In [ ]:
import duckdb, pandas as pd, numpy as np

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")

# --- 2a. Entropía histórica de ganadores por entidad (ventana expandida anual, sin fuga) ---
anios = range(2016, 2027)
resultados_entropia = []
for anio_corte in anios:
    entropia_anio = con2.execute(f"""
        WITH conteos AS (
            SELECT entidad_ruc, proveedor_ruc, COUNT(*) AS n_contratos
            FROM base_modelado_v14
            WHERE anio_proceso < {anio_corte} AND entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL
            GROUP BY entidad_ruc, proveedor_ruc
        ),
        totales AS (
            SELECT entidad_ruc, SUM(n_contratos) AS total, COUNT(DISTINCT proveedor_ruc) AS n_proveedores_distintos
            FROM conteos GROUP BY entidad_ruc
        ),
        proporciones AS (
            SELECT c.entidad_ruc, CAST(c.n_contratos AS DOUBLE) / t.total AS p, t.total, t.n_proveedores_distintos
            FROM conteos c JOIN totales t ON c.entidad_ruc = t.entidad_ruc
        )
        SELECT entidad_ruc,
               -SUM(p * LN(p)) AS entropia_ganadores,
               MAX(n_proveedores_distintos) AS n_proveedores_distintos,
               MAX(total) AS n_contratos_previos,
               {anio_corte} AS anio_aplicable
        FROM proporciones
        GROUP BY entidad_ruc
    """).df()
    resultados_entropia.append(entropia_anio)
    print(f"Año {anio_corte}: {len(entropia_anio)} entidades con historial previo")

entropia_expandida = pd.concat(resultados_entropia, ignore_index=True)
# Normalizada (0=un solo proveedor gana siempre, 1=máxima diversidad dado el n de proveedores)
entropia_expandida['entropia_normalizada'] = entropia_expandida['entropia_ganadores'] / np.log(
    entropia_expandida['n_proveedores_distintos'].clip(lower=2))
entropia_expandida.to_parquet('03_resultados/entropia_ganadores_expandido.parquet', index=False)
print("\nentropia_ganadores_expandido:", len(entropia_expandida), "filas")
print(entropia_expandida[['entropia_ganadores','entropia_normalizada','n_proveedores_distintos']].describe())

# --- 2b. Racha actual del proveedor en esa entidad (contemporánea, gaps-and-islands) ---
con2.execute("CREATE OR REPLACE TABLE entropia_ganadores_expandido AS SELECT * FROM entropia_expandida")

base_racha = con2.execute("""
    WITH ordenado AS (
        SELECT ocid, entidad_ruc, proveedor_ruc, fecha_release, anio_proceso,
               ROW_NUMBER() OVER (PARTITION BY entidad_ruc ORDER BY fecha_release) AS rn_entidad,
               ROW_NUMBER() OVER (PARTITION BY entidad_ruc, proveedor_ruc ORDER BY fecha_release) AS rn_proveedor
        FROM base_modelado_v14
        WHERE entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL
    ),
    grupos AS (
        SELECT *, rn_entidad - rn_proveedor AS grupo_id FROM ordenado
    )
    SELECT ocid, entidad_ruc, proveedor_ruc,
           ROW_NUMBER() OVER (PARTITION BY entidad_ruc, grupo_id ORDER BY fecha_release) AS entidad_racha_proveedor_actual
    FROM grupos
""").df()
base_racha.to_parquet('03_resultados/racha_proveedor_entidad.parquet', index=False)
print("\nracha_proveedor_entidad:", len(base_racha), "filas")
print(base_racha['entidad_racha_proveedor_actual'].describe())

# --- Comparación contra fantasmas conocidos (uniendo ambas por ocid / entidad+año) ---
con2.execute("CREATE OR REPLACE TABLE racha_proveedor_entidad AS SELECT * FROM base_racha")
comparacion = con2.execute("""
    SELECT b.proveedor_es_fantasma,
           AVG(e.entropia_normalizada) AS entropia_norm_promedio,
           MEDIAN(e.entropia_normalizada) AS entropia_norm_mediana,
           AVG(r.entidad_racha_proveedor_actual) AS racha_promedio,
           MEDIAN(r.entidad_racha_proveedor_actual) AS racha_mediana,
           COUNT(*) AS n
    FROM base_modelado_v14 b
    JOIN entropia_ganadores_expandido e ON b.entidad_ruc = e.entidad_ruc AND b.anio_proceso = e.anio_aplicable
    JOIN racha_proveedor_entidad r ON b.ocid = r.ocid
    GROUP BY b.proveedor_es_fantasma
""").df()
print("\nComparación fantasma vs. resto:")
print(comparacion)

In [ ]:
import duckdb, pandas as pd

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")

# Mercado = categoria_principal (con fallback a tipo_contratacion_canonico si es nulo)
con2.execute("""
    CREATE OR REPLACE TABLE base_mercado AS
    SELECT *, COALESCE(categoria_principal, tipo_contratacion_canonico) AS mercado
    FROM base_modelado_v14
    WHERE proveedor_ruc IS NOT NULL
""")

# Año de primera aparición de cada proveedor en cada mercado (usando TODO el historial, sirve como referencia)
primera_aparicion = con2.execute("""
    SELECT proveedor_ruc, mercado, MIN(anio_proceso) AS anio_primera_aparicion
    FROM base_mercado
    GROUP BY proveedor_ruc, mercado
""").df()
con2.execute("CREATE OR REPLACE TABLE primera_aparicion AS SELECT * FROM primera_aparicion")

# Feature sin fuga: en el momento del proceso, ¿es su primer año en ese mercado? (comparando anio_proceso == anio_primera_aparicion)
base_entrada = con2.execute("""
    SELECT b.ocid, b.proveedor_ruc, b.mercado, b.anio_proceso,
           (b.anio_proceso = p.anio_primera_aparicion) AS proveedor_es_nuevo_en_mercado,
           b.anio_proceso - COALESCE(
               (SELECT MIN(anio_primera_aparicion) FROM primera_aparicion p2
                WHERE p2.proveedor_ruc = b.proveedor_ruc AND p2.anio_primera_aparicion < b.anio_proceso),
               b.anio_proceso
           ) AS anios_desde_primera_actividad_general
    FROM base_mercado b
    JOIN primera_aparicion p ON b.proveedor_ruc = p.proveedor_ruc AND b.mercado = p.mercado
""").df()
base_entrada.to_parquet('03_resultados/entrada_mercado.parquet', index=False)
print("Filas:", len(base_entrada))
print("% proveedor_es_nuevo_en_mercado:", base_entrada['proveedor_es_nuevo_en_mercado'].mean())

# Comparación contra fantasmas conocidos
con2.execute("CREATE OR REPLACE TABLE entrada_mercado AS SELECT * FROM base_entrada")
comparacion = con2.execute("""
    SELECT b.proveedor_es_fantasma,
           AVG(CAST(e.proveedor_es_nuevo_en_mercado AS INT)) AS pct_nuevo_en_mercado,
           AVG(e.anios_desde_primera_actividad_general) AS anios_desde_inicio_promedio,
           MEDIAN(e.anios_desde_primera_actividad_general) AS anios_desde_inicio_mediana,
           COUNT(*) AS n
    FROM base_modelado_v14 b
    JOIN entrada_mercado e ON b.ocid = e.ocid
    GROUP BY b.proveedor_es_fantasma
""").df()
print("\nComparación fantasma vs. resto:")
print(comparacion)

Bloque A — Ley de Benford sobre montos, por proveedor (ventana expandida)

Idea: en una serie de montos "naturales" (no manipulados), el primer dígito sigue la distribución de Benford (1 aparece ~30% de las veces, 9 solo ~4.6%). Un proveedor cuyos montos históricos se desvían mucho de esa distribución es indicio de cifras ajustadas/artificiales. Se calcula solo para proveedores con suficiente historial (≥10 contratos previos) para que el estadístico tenga sentido.

In [ ]:
import duckdb, pandas as pd, numpy as np

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")

# Distribución esperada de Benford para el primer dígito
benford_esperado = {d: np.log10(1 + 1/d) for d in range(1, 10)}

anios = range(2016, 2027)
resultados_benford = []
for anio_corte in anios:
    datos = con2.execute(f"""
        SELECT proveedor_ruc,
               CAST(SUBSTRING(CAST(CAST(monto_final AS BIGINT) AS VARCHAR), 1, 1) AS INT) AS primer_digito
        FROM base_modelado_v14
        WHERE anio_proceso < {anio_corte} AND monto_final > 0 AND proveedor_ruc IS NOT NULL
    """).df()
    conteo = datos.groupby(['proveedor_ruc', 'primer_digito']).size().unstack(fill_value=0)
    conteo = conteo.reindex(columns=range(1,10), fill_value=0)
    totales = conteo.sum(axis=1)
    elegibles = totales[totales >= 10].index
    conteo = conteo.loc[elegibles]
    totales = totales.loc[elegibles]

    observado = conteo.div(totales, axis=0)
    esperado = pd.Series(benford_esperado)
    # MAD de Nigrini: promedio de |observado - esperado| por dígito
    mad_benford = (observado - esperado).abs().mean(axis=1)
    chi2 = (((observado - esperado)**2 / esperado).sum(axis=1) * totales)

    resultado_anio = pd.DataFrame({
        'proveedor_ruc': elegibles,
        'mad_benford': mad_benford.values,
        'chi2_benford': chi2.values,
        'n_contratos_previos': totales.values,
        'anio_aplicable': anio_corte
    })
    resultados_benford.append(resultado_anio)
    print(f"Año {anio_corte}: {len(resultado_anio)} proveedores elegibles (>=10 contratos previos)")

benford_expandido = pd.concat(resultados_benford, ignore_index=True)
benford_expandido.to_parquet('03_resultados/benford_expandido.parquet', index=False)
print("\nbenford_expandido:", len(benford_expandido), "filas")
print(benford_expandido[['mad_benford','chi2_benford']].describe())

# Comparación contra fantasmas conocidos
con2.execute("CREATE OR REPLACE TABLE benford_expandido AS SELECT * FROM benford_expandido")
comparacion = con2.execute("""
    SELECT b.proveedor_es_fantasma,
           AVG(f.mad_benford) AS mad_promedio,
           MEDIAN(f.mad_benford) AS mad_mediana,
           AVG(f.chi2_benford) AS chi2_promedio,
           COUNT(*) AS n
    FROM base_modelado_v14 b
    JOIN benford_expandido f ON b.proveedor_ruc = f.proveedor_ruc AND b.anio_proceso = f.anio_aplicable
    GROUP BY b.proveedor_es_fantasma
""").df()
print("\nComparación fantasma vs. resto:")
print(comparacion)

In [ ]:
import duckdb, pandas as pd

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")

# Para cada par entidad-proveedor, ordenar contratos por fecha y comparar cada monto contra el primero del par
escalamiento = con2.execute("""
    WITH ordenado AS (
        SELECT ocid, entidad_ruc, proveedor_ruc, monto_final, fecha_release,
               ROW_NUMBER() OVER (PARTITION BY entidad_ruc, proveedor_ruc ORDER BY fecha_release) AS orden_contrato,
               FIRST_VALUE(monto_final) OVER (PARTITION BY entidad_ruc, proveedor_ruc ORDER BY fecha_release) AS primer_monto_par
        FROM base_modelado_v14
        WHERE entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL AND monto_final > 0
    )
    SELECT ocid, entidad_ruc, proveedor_ruc, orden_contrato, monto_final, primer_monto_par,
           monto_final / primer_monto_par AS ratio_escalamiento
    FROM ordenado
    WHERE orden_contrato >= 2
""").df()
escalamiento.to_parquet('03_resultados/escalamiento_montos.parquet', index=False)
print("Filas (contratos a partir del 2do del par):", len(escalamiento))
print(escalamiento['ratio_escalamiento'].describe())

for umbral in [3, 5, 10, 20]:
    pct = (escalamiento['ratio_escalamiento'] >= umbral).mean()
    print(f"% de contratos con ratio >= {umbral}x el primer monto del par: {pct*100:.2f}%")

# Comparación contra fantasmas conocidos
con2.execute("CREATE OR REPLACE TABLE escalamiento_montos AS SELECT * FROM escalamiento")
comparacion = con2.execute("""
    SELECT b.proveedor_es_fantasma,
           AVG(e.ratio_escalamiento) AS ratio_promedio,
           MEDIAN(e.ratio_escalamiento) AS ratio_mediana,
           AVG(CASE WHEN e.ratio_escalamiento >= 5 THEN 1.0 ELSE 0.0 END) AS pct_escalada_5x,
           COUNT(*) AS n
    FROM base_modelado_v14 b
    JOIN escalamiento_montos e ON b.ocid = e.ocid
    GROUP BY b.proveedor_es_fantasma
""").df()
print("\nComparación fantasma vs. resto (solo contratos que son el 2do+ del par):")
print(comparacion)

In [ ]:
import duckdb, pandas as pd, numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")
con2.execute("CREATE OR REPLACE TABLE perfil_dominancia_expandido AS SELECT * FROM read_parquet('03_resultados/perfil_dominancia_expandido.parquet')")
con2.execute("CREATE OR REPLACE TABLE winrate_expandido AS SELECT * FROM read_parquet('03_resultados/winrate_expandido.parquet')")
con2.execute("CREATE OR REPLACE TABLE competidor_habitual AS SELECT * FROM read_parquet('03_resultados/competidor_habitual.parquet')")
con2.execute("CREATE OR REPLACE TABLE entrada_mercado AS SELECT * FROM read_parquet('03_resultados/entrada_mercado.parquet')")
con2.execute("CREATE OR REPLACE TABLE benford_expandido AS SELECT * FROM read_parquet('03_resultados/benford_expandido.parquet')")

base_m5 = con2.execute("""
    SELECT b.*,
           d.dominancia_maxima_historica, d.dominancia_promedio_historica, d.n_pares_dominancia_alta,
           w.win_rate, w.loss_rate, w.n_participaciones, w.perdedor_recurrente,
           c.concentracion_rival_habitual, c.n_rivales_distintos,
           e.proveedor_es_nuevo_en_mercado, e.anios_desde_primera_actividad_general,
           f.mad_benford
    FROM base_modelado_v14 b
    LEFT JOIN perfil_dominancia_expandido d ON b.proveedor_ruc = d.proveedor_ruc AND b.anio_proceso = d.anio_aplicable
    LEFT JOIN winrate_expandido w ON b.proveedor_ruc = w.oferente_ruc AND b.anio_proceso = w.anio_aplicable
    LEFT JOIN competidor_habitual c ON b.proveedor_ruc = c.proveedor_ruc AND b.anio_proceso = c.anio_aplicable
    LEFT JOIN entrada_mercado e ON b.ocid = e.ocid
    LEFT JOIN benford_expandido f ON b.proveedor_ruc = f.proveedor_ruc AND b.anio_proceso = f.anio_aplicable
""").df()

for col in ['dominancia_maxima_historica','dominancia_promedio_historica','n_pares_dominancia_alta',
            'win_rate','loss_rate','n_participaciones','concentracion_rival_habitual','n_rivales_distintos',
            'anios_desde_primera_actividad_general','mad_benford']:
    base_m5[f'{col}_desconocido'] = base_m5[col].isna().astype(int)
    base_m5[col] = base_m5[col].fillna(-1)
base_m5['perdedor_recurrente'] = base_m5['perdedor_recurrente'].fillna(False).astype(int)
base_m5['proveedor_es_nuevo_en_mercado'] = base_m5['proveedor_es_nuevo_en_mercado'].fillna(False).astype(int)

train_mask = base_m5['anio_proceso'].between(2015, 2017)
test_mask  = base_m5['anio_proceso'].between(2018, 2019)
print("Train (2015-2017):", train_mask.sum(), "filas |", base_m5.loc[train_mask,'proveedor_es_fantasma'].sum(), "fantasmas")
print("Test  (2018-2019):", test_mask.sum(), "filas |", base_m5.loc[test_mask,'proveedor_es_fantasma'].sum(), "fantasmas")

features = ['desviacion_precio_log_z_sinfuga','n_oferentes','proveedor_unico','proveedor_unico_desconocido',
            'dominancia_maxima_historica','dominancia_promedio_historica','n_pares_dominancia_alta',
            'win_rate','loss_rate','n_participaciones','perdedor_recurrente',
            'concentracion_rival_habitual','n_rivales_distintos',
            'proveedor_es_nuevo_en_mercado','anios_desde_primera_actividad_general','mad_benford',
            'dominancia_maxima_historica_desconocido','win_rate_desconocido',
            'concentracion_rival_habitual_desconocido','mad_benford_desconocido']
features = [f for f in features if f in base_m5.columns]

X_train, y_train = base_m5.loc[train_mask, features], base_m5.loc[train_mask, 'proveedor_es_fantasma']
X_test,  y_test  = base_m5.loc[test_mask, features],  base_m5.loc[test_mask, 'proveedor_es_fantasma']

scale = (y_train==0).sum() / max((y_train==1).sum(), 1)
modelo3_v5 = xgb.XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=scale,
                                eval_metric='aucpr', random_state=42)
modelo3_v5.fit(X_train, y_train)

scores_test = modelo3_v5.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)
print(f"\\nAUC-ROC Modelo 3 v5: {auc:.4f}  (v4: 0.7796)")
print(f"PR-AUC Modelo 3 v5: {pr_auc:.4f}  (v4: 0.0003, línea base: {y_test.mean():.5f})")

top_n_full = base_m5.loc[test_mask].assign(score=scores_test).sort_values('score', ascending=False).reset_index(drop=True)
top_n_full['percentil'] = (top_n_full.index + 1) / len(top_n_full)
fantasmas_test = top_n_full[top_n_full['proveedor_es_fantasma'] == True]
print(f"\\nPercentil mediano de los fantasmas: {fantasmas_test['percentil'].median():.4f}  (v4: 0.0720)")
print(f"Percentil promedio: {fantasmas_test['percentil'].mean():.4f}  (v4: 0.2205)")

resumen_proveedor = top_n_full.groupby('proveedor_ruc').agg(
    score_maximo=('score','max'), percentil_minimo=('percentil','min'),
    es_fantasma=('proveedor_es_fantasma','max')).reset_index().sort_values('score_maximo', ascending=False)
n_fantasmas_unicos = resumen_proveedor['es_fantasma'].sum()
for k in [10, 50, 100, 500, 1000]:
    capturados = resumen_proveedor.head(k)['es_fantasma'].sum()
    print(f"Top {k} proveedores: {capturados} de {n_fantasmas_unicos} fantasmas únicos  (v4 en todos: 0)")

importancias = pd.Series(modelo3_v5.feature_importances_, index=features).sort_values(ascending=False)
print("\\nImportancia de variables:\\n", importancias)

In [ ]:
import os, pandas as pd

carpeta = r'C:\SERCOP_FINAL\05_documentos\directorio_companias'
print(os.listdir(carpeta))

In [ ]:
import pandas as pd

archivo = r'C:\SERCOP_FINAL\05_documentos\directorio_companias.xlsx'
df_directorio = pd.read_excel(archivo, nrows=20)
print("Columnas:", df_directorio.columns.tolist())
print("\nMuestra:")
print(df_directorio.head(10))
print("\nTipos:")
print(df_directorio.dtypes)

# Tamaño total del archivo (para saber si conviene leerlo completo de una)
import openpyxl
wb = openpyxl.load_workbook(archivo, read_only=True)
ws = wb.active
print("\nFilas totales (aprox):", ws.max_row, "| Columnas:", ws.max_column)
print("Hojas del archivo:", wb.sheetnames)